# Metrics Table by Dataset and Action

This notebook loads `results/all_models_metrics_long.csv`, filters rows by `dataset` and `action_filter`, and displays a styled table where the best value for each metric is **bold** and the second best is <span style="text-decoration: underline;">underlined</span>.

`APD` is treated as higher-is-better. The other displayed metrics are treated as lower-is-better.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

CSV_PATH = Path('/home/fagnelli/diffusion_hands/results/all_models_metrics_long.csv')
METRIC_COLUMNS = [
    'APD',
    'ADE',
    'FDE',
    'MMADE',
    'MMFDE',
    'CMD',
    'FID',
]
HIGHER_IS_BETTER = {'APD'}

df = pd.read_csv(CSV_PATH)
df['action_filter'] = df['action_filter'].fillna('')


def build_metrics_table(dataset, action_filter=''):
    action_filter = '' if action_filter is None else str(action_filter)

    filtered = df.loc[
        (df['status'] == 'ok')
        & (df['dataset'] == dataset)
        & (df['action_filter'] == action_filter),
        ['model', *METRIC_COLUMNS],
    ].copy()

    if filtered.empty:
        raise ValueError(
            f'No rows found for dataset={dataset!r} and action_filter={action_filter!r}.'
        )

    filtered = filtered.sort_values('model').reset_index(drop=True)

    def highlight_best_and_second(series):
        styles = ['' for _ in series]
        ascending = series.name not in HIGHER_IS_BETTER
        ranked = series.dropna().sort_values(ascending=ascending, kind='stable')
        unique_values = ranked.drop_duplicates().tolist()

        if unique_values:
            best_value = unique_values[0]
            for idx in series.index[series.eq(best_value)]:
                styles[idx] = 'font-weight: bold;'

        if len(unique_values) > 1:
            second_value = unique_values[1]
            second_style = 'text-decoration: underline; text-underline-offset: 0.15em;'
            for idx in series.index[series.eq(second_value)]:
                styles[idx] = (styles[idx] + ' ' + second_style).strip()

        return styles

    styled = (
        filtered.style.format({metric: '{:.3f}' for metric in METRIC_COLUMNS})
        .apply(highlight_best_and_second, subset=METRIC_COLUMNS, axis=0)
    )
    return filtered, styled


In [ ]:
# Set the filters here.
# Examples:
#   dataset_filter = 'assembly'; action_filter = 'attempt'
#   dataset_filter = 'h2o'; action_filter = ''

dataset_filter = 'assembly'
action_filter = 'attempt'

table, styled_table = build_metrics_table(dataset_filter, action_filter)
print(f"dataset={dataset_filter!r}, action_filter={action_filter!r}, rows={len(table)}")
display(styled_table)


In [ ]:
# Optional helper: inspect available filter values.
available_filters = (
    df[['dataset', 'action_filter']]
    .drop_duplicates()
    .sort_values(['dataset', 'action_filter'])
    .reset_index(drop=True)
)
display(available_filters)
